# COCO-format annotation files for Pascal VOC, Cityscapes and KITTI

The inference scripts, `label_spreading/build_soft_gt.py` and the label-error
study read each dataset's images and boxes from COCO-format JSONs. This
notebook creates them from the raw annotations under `DATA_ROOT` (`~/data`):

```
VOC/VOCdevkit/VOC2012/{JPEGImages,Annotations,ImageSets/Main/trainval.txt}
Cityscapes/{train,val}/{images,labels,our_labels}/
Cityscapes/trainval/images/   (train + val images, created below as symlinks)
KITTI/train/{images,labels,our_labels}/
```

`labels/` holds the datasets' official annotations and `our_labels/` the boxes
of the soft-label benchmark, both as YOLO `.txt` files. For Cityscapes and
KITTI this gives two files each:

- `soft_instances_*.json` (from `our_labels/`): the soft-label benchmark's
  boxes, used for inference and soft-GT harmonization,
- `instances_*.json` (from `labels/`): the official annotations, used as the
  original ground truth in the label-error study.

For Pascal VOC, `instances_trainval.json` is built from the official XML
annotations of the detection trainval split and serves both purposes.

In [1]:
import os
import json
from PIL import Image

DATA_ROOT = os.path.expanduser("~/data")

# Cityscapes: train and val use disjoint cities, merged into one trainval split.
# Images are symlinked into trainval/images (the inference input); labels are read
# from the train/ and val/ directories directly.
CITYSCAPES_ROOT = f"{DATA_ROOT}/Cityscapes"
CITYSCAPES_TRAINVAL_IMAGES = f"{CITYSCAPES_ROOT}/trainval/images"
os.makedirs(CITYSCAPES_TRAINVAL_IMAGES, exist_ok=True)
for split in ["train", "val"]:
    for fname in os.listdir(f"{CITYSCAPES_ROOT}/{split}/images"):
        link = os.path.join(CITYSCAPES_TRAINVAL_IMAGES, fname)
        if not os.path.lexists(link):
            os.symlink(f"{CITYSCAPES_ROOT}/{split}/images/{fname}", link)

city_names = {fname.split("_")[0] for fname in os.listdir(CITYSCAPES_TRAINVAL_IMAGES)}

# city names sorted alphabetically, mapped to integers starting from 1, giving numeric
# image ids: city_000012_000001.png -> <city id>000012000001
city_name_to_id = {name: idx+1 for idx, name in enumerate(sorted(city_names))}

In [2]:
def yolo_to_coco(img_dir, label_dirs, class_names):
    """COCO-format dict from YOLO .txt labels; each image's label file is looked up in
    label_dirs (a list of directories, searched in order)."""
    images = []
    annotations = []
    categories = []

    # COCO categories
    for i, name in enumerate(class_names):
        categories.append({
            "id": i,
            "name": name,
            "supercategory": "none"
        })

    ann_id = 1

    img_files = [f for f in os.listdir(img_dir)
                 if f.endswith((".jpg", ".png", ".jpeg"))]

    for img_file in img_files:
        img_path = os.path.join(img_dir, img_file)

        if not "Cityscapes" in img_dir:  # Cityscapes has non-numeric filenames
            img_id = int(os.path.splitext(img_file)[0])  # assumes filename is numeric, which is not the case for Cityscapes, but for KITTI and PascalVOC
        else:
            city = img_file.split("_")[0]
            city_id = city_name_to_id[city]

            numeric_img_file = img_file.replace(city, str(city_id))  # remove city name from filename
            img_id = int(numeric_img_file.replace("_", "").split(".")[0])  # remove underscores and .png and convert to int

        label_name = os.path.splitext(img_file)[0] + ".txt"
        label_path = next((os.path.join(d, label_name) for d in label_dirs
                           if os.path.exists(os.path.join(d, label_name))), "")

        # read image size
        with Image.open(img_path) as img:
            w, h = img.size

        images.append({
            "id": img_id,
            "file_name": img_file,
            "width": w,
            "height": h
        })

        # read YOLO labels
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                lines = f.readlines()

            for line in lines:
                cls, xc, yc, bw, bh = map(float, line.strip().split())

                cls = int(cls)

                # YOLO → pixel xyxy
                x_center = xc * w
                y_center = yc * h
                box_w = bw * w
                box_h = bh * h

                x1 = x_center - box_w / 2
                y1 = y_center - box_h / 2

                annotations.append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": cls,
                    "bbox": [x1, y1, box_w, box_h],
                    "area": box_w * box_h,
                    "iscrowd": 0
                })

                ann_id += 1

    return {
        "images": images,
        "annotations": annotations,
        "categories": categories
    }

In [3]:
import xml.etree.ElementTree as ET

def voc_xml_to_coco(img_dir, xml_dir, class_names, id_list=None):
    images = []
    annotations = []
    categories = [{"id": i, "name": name, "supercategory": "none"} for i, name in enumerate(class_names)]
    name_to_id = {name: i for i, name in enumerate(class_names)}

    if id_list is None:
        id_list = sorted(os.path.splitext(f)[0] for f in os.listdir(xml_dir) if f.endswith(".xml"))

    ann_id = 1
    for stem in id_list:
        tree = ET.parse(os.path.join(xml_dir, stem + ".xml"))
        root = tree.getroot()

        file_name = root.find("filename").text
        size = root.find("size")
        w, h = int(size.find("width").text), int(size.find("height").text)
        img_id = int(stem.replace("_", ""))  # e.g. 2008_000008 -> 2008000008, unique per VOC's year_seqnum filenames

        images.append({"id": img_id, "file_name": file_name, "width": w, "height": h})

        for obj in root.findall("object"):
            name = obj.find("name").text
            if name not in name_to_id:
                continue
            box = obj.find("bndbox")
            xmin, ymin = float(box.find("xmin").text), float(box.find("ymin").text)
            xmax, ymax = float(box.find("xmax").text), float(box.find("ymax").text)
            bw, bh = xmax - xmin, ymax - ymin

            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": name_to_id[name],
                "bbox": [xmin, ymin, bw, bh],
                "area": bw * bh,
                "iscrowd": 0,
            })
            ann_id += 1

    return {"images": images, "annotations": annotations, "categories": categories}

In [4]:
# Pascal VOC -> COCO format from the official XML annotations, restricted to the
# detection trainval split (ImageSets/Main/trainval.txt).

VOC_ROOT = f"{DATA_ROOT}/VOC/VOCdevkit/VOC2012"
image_dir = f"{VOC_ROOT}/JPEGImages"
xml_dir = f"{VOC_ROOT}/Annotations"

with open(f"{VOC_ROOT}/ImageSets/Main/trainval.txt") as f:
    trainval_ids = [line.strip() for line in f if line.strip()]

cat_list=[
        "aeroplane","bicycle","bird","boat","bottle",
        "bus","car","cat","chair","cow",
        "diningtable","dog","horse","motorbike","person",
        "pottedplant","sheep","sofa","train","tvmonitor"
    ]

coco_dict = voc_xml_to_coco(
            img_dir=image_dir,
            xml_dir=xml_dir,
            class_names=cat_list,
            id_list=trainval_ids,
        )

print(len(coco_dict["images"]), len(coco_dict["annotations"]), len(coco_dict["categories"]))

print(coco_dict["images"][0])

with open(f"{VOC_ROOT}/instances_trainval.json", "w") as f:
    json.dump(coco_dict, f)

11540 31561 20
{'id': 2008000002, 'file_name': '2008_000002.jpg', 'width': 500, 'height': 375}


In [5]:
# KITTI -> COCO format from YOLO labels (train split)
KITTI_ROOT = f"{DATA_ROOT}/KITTI/train"
cat_list = ["car", "van", "truck", "pedestrian", "person_sitting", "cyclist", "tram"]

for label_subdir, out_name in [("our_labels", "soft_instances_train.json"), ("labels", "instances_train.json")]:
    coco_dict = yolo_to_coco(img_dir=f"{KITTI_ROOT}/images",
                             label_dirs=[f"{KITTI_ROOT}/{label_subdir}"],
                             class_names=cat_list)
    print(out_name, len(coco_dict["images"]), len(coco_dict["annotations"]), len(coco_dict["categories"]))
    with open(f"{KITTI_ROOT}/{out_name}", "w") as f:
        json.dump(coco_dict, f)

soft_instances_train.json 7481 63348 7


instances_train.json 7481 39597 7


In [6]:
# Cityscapes -> COCO format from YOLO labels, train+val merged into one trainval split
cat_list = ["person", "rider", "car", "truck", "bus", "train", "motorcycle", "bicycle"]

for label_subdir, out_name in [("our_labels", "soft_instances_trainval.json"), ("labels", "instances_trainval.json")]:
    coco_dict = yolo_to_coco(img_dir=CITYSCAPES_TRAINVAL_IMAGES,
                             label_dirs=[f"{CITYSCAPES_ROOT}/{split}/{label_subdir}" for split in ["train", "val"]],
                             class_names=cat_list)
    print(out_name, len(coco_dict["images"]), len(coco_dict["annotations"]), len(coco_dict["categories"]))
    with open(f"{CITYSCAPES_ROOT}/trainval/{out_name}", "w") as f:
        json.dump(coco_dict, f)

soft_instances_trainval.json 3475 65856 8


instances_trainval.json 3475 62649 8
